# Tomato Pest Classifier: Experiment A

A small CNN (**2 convolutional layers**) followed by exactly **two Dense layers**.

| Hyperparameter | Experiment A |
|---|---|
| Conv. layers | 2 |
| L2 regularization | 0.0001 |
| Dropout | 0.1 |
| Batch size | 32 |
| Learning rate | 0.001 |
| Maximum epochs | 50 |
| Early stopping patience | 5 |

**Metrics (held-out test set):** Accuracy, macro F1, macro one-vs-rest AUC.

## 1. Mock dataset (pipeline test only)

This creates random-noise images so the code can be tested end to end. Results on it are
meaningless. The cell is skipped if `tomato_pests/` already exists, so it never mixes noise
into a real dataset. For real training, put a public tomato pest dataset (one folder per
class) in `tomato_pests/`.

In [ ]:
import os
import numpy as np
from PIL import Image

DATA_DIR = 'tomato_pests'
classes = ['spider_mites', 'early_blight']

if not os.path.isdir(DATA_DIR):
    for cls in classes:
        cls_dir = os.path.join(DATA_DIR, cls)
        os.makedirs(cls_dir, exist_ok=True)
        for i in range(50):
            img_data = np.random.randint(0, 255, (224, 224, 3), dtype=np.uint8)
            Image.fromarray(img_data).save(os.path.join(cls_dir, f'image_{i}.jpg'))
    print(f"Created mock dataset at '{DATA_DIR}/' with classes {classes}")
else:
    print(f"Using existing dataset at '{DATA_DIR}/'")

## 2. Configuration (Experiment A)

In [ ]:
import numpy as np
import tensorflow as tf
from tensorflow.keras import layers, models, regularizers
from sklearn.metrics import accuracy_score, f1_score, roc_auc_score, classification_report

DATA_DIR = "tomato_pests"
IMG_SIZE = (224, 224)
SEED = 42

# Experiment A hyperparameters
N_CONV = 2
L2 = 1e-4
DROPOUT = 0.1
BATCH = 32
LR = 1e-3
MAX_EPOCHS = 50
PATIENCE = 5

tf.keras.utils.set_random_seed(SEED)

## 3. Load data (70% train / 15% validation / 15% test)

The split is done per image, not per batch, so validation and test sets are never empty
on small datasets.

In [ ]:
raw = tf.keras.utils.image_dataset_from_directory(
    DATA_DIR, image_size=IMG_SIZE, batch_size=None,
    label_mode="int", shuffle=False)
class_names = raw.class_names
n_classes = len(class_names)
n_total = int(raw.cardinality().numpy())

# One fixed shuffle so the three splits never overlap
raw = raw.shuffle(n_total, seed=SEED, reshuffle_each_iteration=False)
n_train = int(0.70 * n_total)
n_val = int(0.15 * n_total)

AUTOTUNE = tf.data.AUTOTUNE
train_ds = (raw.take(n_train).shuffle(1000, seed=SEED)
            .batch(BATCH).prefetch(AUTOTUNE))
val_ds = raw.skip(n_train).take(n_val).batch(BATCH).prefetch(AUTOTUNE)
test_ds = raw.skip(n_train + n_val).batch(BATCH).prefetch(AUTOTUNE)

print("Classes:", class_names)
print("Images -> train:", n_train, "| val:", n_val, "| test:", n_total - n_train - n_val)

## 4. Build the model

Rescaling and augmentation -> 2 x (Conv2D + MaxPool) -> global average pooling ->
Dropout(0.1) -> **Dense(256, relu)** -> **Dense(n_classes, softmax)**.
L2 (0.0001) is applied to the conv and Dense kernels.

In [ ]:
reg = regularizers.l2(L2)

augment = tf.keras.Sequential([
    layers.RandomFlip("horizontal_and_vertical"),
    layers.RandomRotation(0.15),
    layers.RandomZoom(0.1),
])

inputs = layers.Input(shape=IMG_SIZE + (3,))
x = augment(inputs)
x = layers.Rescaling(1.0 / 255)(x)

filters = [32, 64]                       # one entry per conv layer (N_CONV = 2)
for f in filters[:N_CONV]:
    x = layers.Conv2D(f, 3, padding="same", activation="relu",
                      kernel_regularizer=reg)(x)
    x = layers.MaxPooling2D()(x)

x = layers.GlobalAveragePooling2D()(x)
x = layers.Dropout(DROPOUT)(x)
x = layers.Dense(256, activation="relu", kernel_regularizer=reg)(x)                    # Dense layer 1
outputs = layers.Dense(n_classes, activation="softmax", kernel_regularizer=reg)(x)     # Dense layer 2
model = models.Model(inputs, outputs)

model.compile(
    optimizer=tf.keras.optimizers.Adam(LR),
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"])
model.summary()

## 5. Train (max 50 epochs, early stopping patience 5)

In [ ]:
callbacks = [
    tf.keras.callbacks.EarlyStopping(monitor="val_loss", patience=PATIENCE,
                                     restore_best_weights=True),
]
history = model.fit(train_ds, validation_data=val_ds,
                    epochs=MAX_EPOCHS, callbacks=callbacks)

## 6. Evaluate on the held-out test set

In [ ]:
y_true = np.concatenate([y.numpy() for _, y in test_ds])
y_prob = model.predict(test_ds)
y_pred = y_prob.argmax(axis=1)

acc = accuracy_score(y_true, y_pred)
f1 = f1_score(y_true, y_pred, average="macro", zero_division=0)
if n_classes == 2:
    auc = roc_auc_score(y_true, y_prob[:, 1])
else:
    auc = roc_auc_score(y_true, y_prob, multi_class="ovr", average="macro",
                        labels=list(range(n_classes)))

print(f"Accuracy        : {acc:.4f}")
print(f"F1 (macro)      : {f1:.4f}")
print(f"AUC (OvR macro) : {auc:.4f}\n")
print(classification_report(y_true, y_pred, target_names=class_names, zero_division=0))

## 7. Save the model

In [ ]:
model.save("tomato_pest_model_expA.keras")